# Stage 2 — FULL projection (Qwen3-32B) on Algoverse A100

**Not the pilot.** Project **all** normalized trajs (~198), all 64 layers, with `--activations-npz`.

**Machine dies in ~48h and wipes the disk.** Checkpoint to Google Drive / GitHub continuously.

**You need uploaded:**
1. `value_axis_32b.npy`
2. `axis_manifest_32b.json` (optional but keep it)
3. Zip of **all** normalized traj JSONs (no need to subsample; skip `ingest_manifest.json`)

**ETA:** often ~1–2 days wall-clock for ~200 trajs (step-count dominated). Start ASAP; resume is built in.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'Need GPU'
print(torch.cuda.get_device_name(0))
print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

In [ ]:
import os
REPO = os.path.expanduser('~/failure_prediction_research')
# If JupyterHub home differs, set REPO explicitly, e.g. '/home/nrja-024b/failure_prediction_research'
if not os.path.isdir(REPO):
    !git clone https://github.com/abdelmagid07/failure_prediction_research.git {REPO}
else:
    %cd {REPO}
    !git pull
%cd {REPO}
!pip install -q -e stage1 -e stage2
!pip install -q pyarrow pandas scikit-learn matplotlib

In [ ]:
# Prefer Drive for outputs (survives machine wipe). If Drive mount fails, use local OUT_DIR
# and `rclone`/`scp`/download often.
from pathlib import Path

try:
    from google.colab import drive  # may not exist on JupyterHub
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/failure_prediction_research/stage2_full_32b')
except Exception as e:
    print('No Colab drive.mount:', e)
    print('Using local checkpoint dir — COPY OFF MACHINE often.')
    DRIVE_ROOT = Path.home() / 'stage2_full_32b_checkpoints'

OUT_DIR = DRIVE_ROOT / 'outputs'
ACT_DIR = DRIVE_ROOT / 'activations'
OUT_DIR.mkdir(parents=True, exist_ok=True)
ACT_DIR.mkdir(parents=True, exist_ok=True)
print('OUT_DIR', OUT_DIR)
print('ACT_DIR', ACT_DIR)

In [ ]:
PRIMARY_LAYER = 49
MODEL = 'Qwen/Qwen3-32B'
N_LAYERS = 64
SAVE_ACTIVATIONS = True  # required for Stage-5 probes

print('PRIMARY_LAYER', PRIMARY_LAYER)
print('SAVE_ACTIVATIONS', SAVE_ACTIVATIONS)

## Upload axis + FULL normalized zip

Three separate uploads (or copy from Drive if you already synced them there).

In [ ]:
import json, shutil, zipfile
from pathlib import Path
import numpy as np

REPO = Path(os.path.expanduser('~/failure_prediction_research'))
AXIS_DIR = REPO / 'stage1' / 'data'
NORM_DIR = REPO / 'stage2' / 'data' / 'normalized_full'
AXIS_DIR.mkdir(parents=True, exist_ok=True)
NORM_DIR.mkdir(parents=True, exist_ok=True)

# --- Option A: Jupyter upload widgets (works on many JupyterHubs) ---
try:
    from google.colab import files as uploader
except Exception:
    uploader = None

def take_upload(prompt, dest: Path, suffix: str):
    print(prompt)
    if uploader is not None:
        up = uploader.upload()
        assert len(up) == 1, list(up)
        name = next(iter(up))
        assert Path(name).suffix.lower() == suffix.lower(), name
        if dest.exists():
            dest.unlink()
        shutil.move(name, dest)
    else:
        # Manual path: put the file on the machine first, then set SRC
        src = Path(input(f'Path to {dest.name}: ').strip())
        assert src.exists() and src.suffix.lower() == suffix.lower(), src
        shutil.copy2(src, dest)
    print('saved', dest)
    return dest

AXIS = take_upload('1/3 value_axis_32b.npy', AXIS_DIR / 'value_axis_32b.npy', '.npy')
axis = np.load(AXIS)
assert axis.shape == (64, 5120), axis.shape

MANIFEST = take_upload('2/3 axis_manifest_32b.json', AXIS_DIR / 'axis_manifest_32b.json', '.json')
print(json.loads(MANIFEST.read_text()).get('primary_layer'),
      json.loads(MANIFEST.read_text()).get('primary_auroc'))

print('3/3 normalized traj zip')
if uploader is not None:
    z_up = uploader.upload()
    zname = next(iter(z_up))
else:
    zname = input('Path to normalized zip: ').strip()

extract = REPO / 'stage2' / 'data' / '_full_zip_extract'
if extract.exists():
    shutil.rmtree(extract)
extract.mkdir(parents=True)
with zipfile.ZipFile(zname) as z:
    z.extractall(extract)

for p in NORM_DIR.glob('*.json'):
    p.unlink()
n = 0
for p in extract.rglob('*.json'):
    if p.name == 'ingest_manifest.json':
        continue
    d = json.loads(p.read_text())
    if not isinstance(d, dict) or 'steps' not in d or 'outcome' not in d:
        continue
    shutil.copy2(p, NORM_DIR / p.name)
    n += 1
print('normalized trajs:', n, 'in', NORM_DIR)
assert n >= 150, 'Expected ~198 full set — check zip'

In [ ]:
# Outcome sanity (need both classes)
import json
from collections import Counter
from pathlib import Path

NORM_DIR = Path(os.path.expanduser('~/failure_prediction_research')) / 'stage2' / 'data' / 'normalized_full'
outs = []
steps = []
for p in NORM_DIR.glob('*.json'):
    d = json.loads(p.read_text())
    outs.append(int(d['outcome']))
    steps.append(d.get('n_steps') or len(d.get('steps', [])))
print('n', len(outs), 'outcomes', Counter(outs))
print('steps mean/median/max', round(sum(steps)/len(steps),1), sorted(steps)[len(steps)//2], max(steps))
assert 0 in outs and 1 in outs

## Project (long GPU job)

Checkpoints after **each traj** to `OUT_DIR` / `ACT_DIR`. Re-run this cell to **resume**.

Keep this browser tab from sleeping if possible; process may continue if kernel stays alive.

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO = Path(os.path.expanduser('~/failure_prediction_research'))
NORM_DIR = REPO / 'stage2' / 'data' / 'normalized_full'
AXIS = REPO / 'stage1' / 'data' / 'value_axis_32b.npy'
PROJ = OUT_DIR / 'projections_full.parquet'
ACT_NPZ = ACT_DIR / 'agentic_meanpool_32b_full.npz'

cmd = [
    sys.executable, '-u', '-m', 'stage2.extract.project_steps',
    '--traj-dir', str(NORM_DIR),
    '--axis-path', str(AXIS),
    '--model', MODEL,
    '--n-layers', str(N_LAYERS),
    '--enable-thinking',
    '--output', str(PROJ),
]
if SAVE_ACTIVATIONS:
    cmd.extend(['--activations-npz', str(ACT_NPZ)])

print('CMD:', ' '.join(cmd), flush=True)
proc = subprocess.Popen(
    cmd, cwd=str(REPO / 'stage2'),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end='', flush=True)
rc = proc.wait()
print('exit', rc, flush=True)
assert rc == 0 and PROJ.exists(), PROJ

## Analyze + probes (CPU)

Can run after projection (even on a laptop if you download parquet/npz). Prefer finishing on this box, then **copy everything off**.

In [ ]:
import os, subprocess, sys, shutil
from pathlib import Path

REPO = Path(os.path.expanduser('~/failure_prediction_research'))
PROJ = OUT_DIR / 'projections_full.parquet'
REPORT = OUT_DIR / 'analysis_report_full'
REPORT.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, '-u', '-m', 'stage2.analyze.run_analyses',
    '--projections', str(PROJ),
    '--output-dir', str(REPORT),
    '--primary-layer', str(PRIMARY_LAYER),
]
print('CMD:', ' '.join(cmd), flush=True)
subprocess.check_call(cmd, cwd=str(REPO / 'stage2'))

ACT_NPZ = ACT_DIR / 'agentic_meanpool_32b_full.npz'
local_act = Path('/tmp/agentic_meanpool_32b_full.npz')
if ACT_NPZ.exists():
    print('copying npz local for faster probe fit...', flush=True)
    shutil.copy2(ACT_NPZ, local_act)
    PROBE = OUT_DIR / 'probe_report_full'
    cmd = [
        sys.executable, '-u', '-m', 'stage2.probes.fit_probes',
        '--activations', str(local_act),
        '--output-dir', str(PROBE),
    ]
    print('CMD:', ' '.join(cmd), flush=True)
    subprocess.check_call(cmd, cwd=str(REPO / 'stage2'))

print('DONE. Copy OUT_DIR off this machine NOW:', OUT_DIR)

In [ ]:
# Zip small artifacts (exclude huge npz if needed)
import zipfile
from pathlib import Path

zip_path = OUT_DIR / 'stage2_full_32b_results_small.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for p in OUT_DIR.rglob('*'):
        if p.is_file() and p.suffix.lower() in {'.json', '.png', '.csv', '.parquet'}:
            z.write(p, p.relative_to(OUT_DIR).as_posix())
print('small zip', zip_path)
print('Also download/sync activations npz separately:', ACT_DIR)